# Kelly Optimization with Quant-Sports

**How much should you actually wager?**

Finding +EV bets is only half the battle. Bet sizing determines whether you grow your bankroll or blow it up. The Kelly Criterion provides the mathematical answer — and Quant-Sports gives you full, fractional, and adaptive Kelly implementations.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from quantitative_sports.core.betting.kelly import (
    KellyCalculator, KellyCalculatorConfig,
    AdaptiveKellyContext,
    BankrollManager, BankrollManagerConfig, ExposureLimits,
    EdgeCalculator,
)
from quantitative_sports.core.betting.odds import Odds
from quantitative_sports.core.risk.portfolio import PortfolioManager, KellyConfig

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
print("Imports successful. Let's optimize bet sizing!")

## 1. Why Bet Sizing Matters

The Kelly Criterion answers: *"What fraction of my bankroll should I bet to maximize long-term growth?"*

$$f^* = \frac{p \cdot b - q}{b}$$

where:
- $p$ = probability of winning
- $q = 1 - p$ = probability of losing
- $b$ = net odds (decimal odds - 1)

Full Kelly maximizes growth rate but has **massive variance**. In practice, fractional Kelly (25-50% of full) is far safer.

## 2. Standard (Full) Kelly

Using `KellyCalculator.compute_kelly()` to find optimal bet fractions.

In [ ]:
kelly = KellyCalculator()

# Example bets with varying edges
example_bets = pd.DataFrame({
    "scenario": ["Small Edge", "Medium Edge", "Large Edge", "Huge Edge", "No Edge", "Negative Edge"],
    "probability": [0.53, 0.57, 0.62, 0.70, 0.50, 0.45],
    "decimal_odds": [1.91, 1.91, 1.91, 1.91, 1.91, 1.91],
})

example_bets["full_kelly"] = example_bets.apply(
    lambda r: kelly.compute_kelly(r["probability"], r["decimal_odds"]), axis=1
)
example_bets["implied_prob"] = 1 / example_bets["decimal_odds"]
example_bets["edge"] = example_bets["probability"] - example_bets["implied_prob"]

example_bets.style.format({"probability": "{:.2f}", "decimal_odds": "{:.2f}", "full_kelly": "{:.4f}", "implied_prob": "{:.3f}", "edge": "{:+.3f}"})

## 3. Fractional Kelly: Half and Quarter

Fractional Kelly reduces variance dramatically while sacrificing only a modest amount of growth.

In [ ]:
for fraction_name, fraction_val in [("Half", 0.50), ("Quarter", 0.25)]:
    example_bets[f"{fraction_name.lower()}_kelly"] = example_bets.apply(
        lambda r: kelly.compute_fractional_kelly(r["probability"], r["decimal_odds"], fraction=fraction_val),
        axis=1,
    )

example_bets[["scenario", "probability", "edge", "full_kelly", "half_kelly", "quarter_kelly"]].style.format({
    "probability": "{:.2f}", "edge": "{:+.3f}", "full_kelly": "{:.4f}", "half_kelly": "{:.4f}", "quarter_kelly": "{:.4f}"
})

## 4. Adaptive Kelly: Sizing with Volatility

Real markets have uncertainty in your probability estimates. `AdaptiveKellyContext` adjusts Kelly sizing based on volatility — bet less when you're less confident.

In [ ]:
prob, odds = 0.57, 1.91

volatility_levels = [0.0, 0.1, 0.2, 0.3, 0.5, 0.7, 0.9]
adaptive_results = []

for vol in volatility_levels:
    ctx = AdaptiveKellyContext(
        probability=prob,
        odds=odds,
        fraction=0.25,  # quarter Kelly base
        volatility=vol,
        max_fraction=0.5,
    )
    adaptive_k = kelly.compute_adaptive_kelly(ctx)
    adaptive_results.append({
        "volatility": vol,
        "volatility_adj": 1.0 - vol,
        "effective_fraction": 0.25 * (1.0 - vol),
        "adaptive_kelly": adaptive_k,
    })

adapt_df = pd.DataFrame(adaptive_results)
adapt_df.style.format({"volatility": "{:.1f}", "volatility_adj": "{:.2f}", "effective_fraction": "{:.4f}", "adaptive_kelly": "{:.6f}"})

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(adapt_df["volatility"], adapt_df["adaptive_kelly"], "o-", linewidth=2, markersize=8, color="steelblue")
ax.fill_between(adapt_df["volatility"], 0, adapt_df["adaptive_kelly"], alpha=0.2, color="steelblue")
ax.set_xlabel("Volatility (Uncertainty Level)")
ax.set_ylabel("Adaptive Kelly Fraction")
ax.set_title("Adaptive Kelly: Bet Less When You're Less Certain")
ax.annotate("High confidence\n= more sizing", xy=(0.05, adapt_df.iloc[0]["adaptive_kelly"]), fontsize=10, color="green")
ax.annotate("Low confidence\n= less sizing", xy=(0.6, adapt_df.iloc[5]["adaptive_kelly"]), fontsize=10, color="red")
plt.tight_layout()
plt.show()

## 5. Bankroll Simulation: Full Kelly vs Quarter Kelly vs Flat Betting

Using `BankrollManager` to simulate 100 bets and compare equity curves.

In [ ]:
np.random.seed(42)

def simulate_bankroll(n_bets, win_prob, decimal_odds, kelly_fraction, initial_bankroll=1000):
    """Simulate bankroll over n bets with given Kelly fraction."""
    kelly_calc = KellyCalculator()
    bm = BankrollManager(initial_bankroll)
    
    bankroll_history = [initial_bankroll]
    full_k = kelly_calc.compute_kelly(win_prob, decimal_odds)
    
    if full_k <= 0:
        return bankroll_history  # no edge, don't bet
    
    for _ in range(n_bets):
        fraction = full_k * kelly_fraction
        bet_size = bm.bankroll * fraction
        bet_size = min(bet_size, bm.bankroll * 0.25)  # cap at 25% for safety
        
        win = np.random.random() < win_prob
        if win:
            profit = bet_size * (decimal_odds - 1)
        else:
            profit = -bet_size
        
        bm._bankroll = bm.update_bankroll(bm.bankroll, profit)
        bankroll_history.append(bm.bankroll)
    
    return bankroll_history

# Simulation parameters
win_prob = 0.55
decimal_odds = 1.91  # -110 American
n_bets = 200
n_sims = 100

strategies = {
    "Full Kelly (1.0)": 1.0,
    "Half Kelly (0.5)": 0.5,
    "Quarter Kelly (0.25)": 0.25,
    "Flat (1% per bet)": None,  # special handling
}

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Left: Mean equity curves with confidence bands
for name, fraction in strategies.items():
    all_curves = []
    for _ in range(n_sims):
        if fraction is None:  # Flat betting
            np.random.seed(_)
            history = [1000]
            bankroll = 1000
            for _ in range(n_bets):
                bet_size = 10  # $10 flat
                win = np.random.random() < win_prob
                bankroll += (bet_size * (decimal_odds - 1)) if win else -bet_size
                history.append(bankroll)
            all_curves.append(history)
        else:
            np.random.seed(_)
            curve = simulate_bankroll(n_bets, win_prob, decimal_odds, fraction)
            all_curves.append(curve)
    
    curves_arr = np.array(all_curves)
    mean_curve = np.mean(curves_arr, axis=0)
    p10 = np.percentile(curves_arr, 10, axis=0)
    p90 = np.percentile(curves_arr, 90, axis=0)
    
    axes[0].plot(mean_curve, label=name, linewidth=2)
    axes[0].fill_between(range(len(mean_curve)), p10, p90, alpha=0.1)

axes[0].set_xlabel("Number of Bets")
axes[0].set_ylabel("Bankroll ($)")
axes[0].set_title("Bankroll Growth: Mean with 80% Confidence Band")
axes[0].legend()
axes[0].set_yscale("log")

# Right: Distribution of final bankrolls
final_bankrolls = {}
for name, fraction in strategies.items():
    finals = []
    for _ in range(n_sims):
        if fraction is None:
            np.random.seed(_ + 1000)
            bankroll = 1000
            for _ in range(n_bets):
                win = np.random.random() < win_prob
                bankroll += (10 * 0.91) if win else -10
            finals.append(bankroll)
        else:
            np.random.seed(_ + 1000)
            curve = simulate_bankroll(n_bets, win_prob, decimal_odds, fraction)
            finals.append(curve[-1])
    final_bankrolls[name] = finals

for name, finals in final_bankrolls.items():
    axes[1].hist(finals, bins=30, alpha=0.5, label=name)

axes[1].set_xlabel("Final Bankroll ($)")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Distribution of Final Bankrolls (100 simulations)")
axes[1].legend()

plt.tight_layout()
plt.show()

# Summary statistics
summary = pd.DataFrame({
    name: {
        "Mean Final": f"${np.mean(v):,.0f}",
        "Median Final": f"${np.median(v):,.0f}",
        "Std Dev": f"${np.std(v):,.0f}",
        "Went Bust (<=0)": f"{sum(1 for x in v if x <= 0)}/{len(v)}",
    }
    for name, v in final_bankrolls.items()
}).T
summary

## 6. Multi-Bet Kelly with Correlations

When betting multiple correlated outcomes (e.g., same-game props), you must account for covariance. `KellyCalculator.compute_kelly_multi_bet()` adjusts fractions.

In [ ]:
# Three correlated bets on the same game
probs = [0.57, 0.55, 0.54]
odds_list = [1.91, 1.91, 1.91]

# Uncorrelated (independent games)
uncorr_kelly = kelly.compute_kelly_multi_bet(probs, odds_list, correlations=None)

# Moderately correlated (same game, different stats)
mod_corr_kelly = kelly.compute_kelly_multi_bet(probs, odds_list, correlations=[0.12, 0.10])

# Highly correlated (same player, different props)
high_corr_kelly = kelly.compute_kelly_multi_bet(probs, odds_list, correlations=[0.35, 0.30])

corr_df = pd.DataFrame({
    "Bet": ["Bet 1 (PTS O)", "Bet 2 (REB O)", "Bet 3 (AST O)"],
    "Probability": probs,
    "Uncorrelated Kelly": uncorr_kelly,
    "Moderate Correlation (r=0.12)": mod_corr_kelly,
    "High Correlation (r=0.35)": high_corr_kelly,
})

corr_df.style.format({"Probability": "{:.2f}", "Uncorrelated Kelly": "{:.4f}", "Moderate Correlation (r=0.12)": "{:.4f}", "High Correlation (r=0.35)": "{:.4f}"})

## 7. Portfolio Management with Multiple Bets

Quant-Sports's `PortfolioManager` tracks exposure, enforces limits, and manages a portfolio of simultaneous bets.

In [ ]:
pm = PortfolioManager(kelly_config=KellyConfig(kelly_fraction=0.25))
pm.set_bankroll(10000)

# Place several bets
bets_to_place = [
    {"player_id": "LeBron", "market": "pts", "line": 27.5, "odds": 1.91, "prob": 0.57, "side": "over", "book": "draftkings"},
    {"player_id": "Curry", "market": "pts", "line": 28.5, "odds": 1.91, "prob": 0.55, "side": "over", "book": "fanduel"},
    {"player_id": "Jokic", "market": "ast", "line": 9.5, "odds": 1.95, "prob": 0.56, "side": "over", "book": "draftkings"},
    {"player_id": "Giannis", "market": "pts", "line": 31.5, "odds": 1.91, "prob": 0.48, "side": "under", "book": "fanduel"},  # -EV, should be rejected
]

for bet in bets_to_place:
    result = pm.place_bet(
        player_id=bet["player_id"],
        market=bet["market"],
        line=bet["line"],
        odds_decimal=bet["odds"],
        win_prob=bet["prob"],
        side=bet["side"],
        book=bet["book"],
    )
    print(f"{bet['player_id']} {bet['market']} {bet['side']} {bet['line']}: {result['status']} (size=${result.get('size', 0):.2f}, reason={result.get('reason', 'OK')})")

# Portfolio summary
print("\n--- Portfolio Summary ---")
summary = pm.get_portfolio_summary()
for k, v in summary.items():
    print(f"{k}: {v}")

In [ ]:
# Settle the bets with simulated outcomes
np.random.seed(42)
outcomes = [1, 1, 0, 0]  # 1=win, 0=loss

for i, outcome in enumerate(outcomes):
    if i < len(pm.bet_history) and pm.bet_history[i]["status"] == "pending":
        pnl = pm.settle_bet(i, outcome)
        print(f"Bet {i} ({pm.bet_history[i]['player_id']}): {'WIN' if outcome else 'LOSS'}, P&L: ${pnl:.2f}")

print(f"\nFinal bankroll: ${pm.bankroll:.2f}")
print(f"Daily P&L: ${pm.daily_pnl:.2f}")

## 8. Exposure Limits and Risk Controls

`BankrollManager` enforces position limits to prevent catastrophic overbetting.

In [ ]:
limits = ExposureLimits(
    max_position_pct=0.10,     # Max 10% on single bet
    max_portfolio_exposure=0.50, # Max 50% total exposure
    max_market_exposure=0.25,    # Max 25% per market
    max_book_exposure=0.30,      # Max 30% per book
)

bm_config = BankrollManagerConfig(
    min_bet_size=5.0,
    max_bet_size=500.0,
    kelly_fraction=0.25,
    exposure_limits=limits,
)

bm = BankrollManager(10000, config=bm_config)

# Test exposure limits
kelly_f = kelly.compute_kelly(0.57, 1.91) * 0.25  # Quarter Kelly
bet_size = bm.compute_bet_size(kelly_f, 1.91, 0.57, current_exposure=0)
print(f"Quarter Kelly fraction: {kelly_f:.4f}")
print(f"Bet size (no exposure): ${bet_size:.2f}")

# With existing exposure
bet_size_with_exposure = bm.compute_bet_size(kelly_f, 1.91, 0.57, current_exposure=4000)
print(f"Bet size ($4K exposure): ${bet_size_with_exposure:.2f}")

# Position limits
clamped = bm.apply_position_limits(1500, bankroll=10000)
print(f"Position limit: $1500 proposed -> ${clamped:.2f} (capped at 10% = $1000)")

# Exposure percentages
exp = bm.get_exposure_percentages(bet_size, current_exposure=0)
print(f"\nExposure metrics: {exp}")

## Key Takeaway

> **Full Kelly maximizes growth but has high volatility; fractional Kelly is the practical choice.**

1. Full Kelly gives the *theoretically* optimal growth rate, but 50% drawdowns are common
2. Quarter Kelly (25%) sacrifices ~15% of growth for ~50% less variance
3. `AdaptiveKellyContext` automatically reduces sizing in uncertain conditions
4. `compute_kelly_multi_bet()` adjusts for correlations between bets
5. `PortfolioManager` and `BankrollManager` enforce hard limits to prevent ruin
6. **Never bet more than you can afford to lose** — Kelly is a guide, not a guarantee